# Gradient Descent: Linear Regression

This notebook shows how gradient descent updates weights to fit a straight line.
We'll compare training with no updates vs. training with SGD.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, '../src')

from neura_command import Sequential, Dense, SGD, MSELoss

np.random.seed(42)


## 1. Generate Data

We'll fit a line `y = 3x + 0.5` with added noise.


In [ ]:
rng = np.random.default_rng(42)
X = rng.uniform(-1, 1, size=(120, 1))
y = 3 * X + 0.5 + rng.normal(0, 0.2, size=X.shape)


## 2. Build a Single-Layer Model


In [ ]:
model = Sequential(
    layers=[Dense(1, 1)],
    optimizer=SGD(learning_rate=0.1),
    loss=MSELoss(),
)

initial_line_x = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
initial_line_y = model.predict(initial_line_x)
initial_loss = model.loss.compute(y, model.predict(X))


## 3. Train with Gradient Descent


In [ ]:
def train_model(model, X, y, epochs=200):
    model.train_mode()
    losses = []
    for _ in range(epochs):
        preds = model.forward(X, training=True)
        loss = model.loss.compute(y, preds)
        losses.append(float(loss))
        grad = model.loss.derivative(y, preds)
        model.backward(grad)
        model.optimizer.step(model.layers)
    return losses

epochs = 200
losses = train_model(model, X, y, epochs=epochs)
baseline_losses = [float(initial_loss)] * epochs


## 4. Visualize Fit and Loss


In [ ]:
trained_line_y = model.predict(initial_line_x)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X, y, alpha=0.6, label='Data')
plt.plot(initial_line_x, initial_line_y, color='gray', linestyle='--', label='Initial line')
plt.plot(initial_line_x, trained_line_y, color='red', label='Trained line')
plt.title('Linear Fit with Gradient Descent')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(losses, label='Gradient Descent')
plt.plot(baseline_losses, linestyle='--', label='No Updates')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()

plt.tight_layout()
plt.show()
